# Phase 3 — Cleveland Corral exploratory dynamics

This restart-and-run notebook explains and inspects the reproducible Phase 3 build. It uses only Phase 2 quality-flagged, explicitly segmented records; fixed PST remains UTC−08:00 year-round. It does **not** interpolate, splice successors, forecast, fit ARIMA/ARIMAX models, detect changepoints, or make causal claims.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = Path.cwd().parent
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), 'Run this notebook inside the project.'
TABLES = PROJECT_ROOT / 'reports/tables/phase3'
FIGURES = PROJECT_ROOT / 'reports/figures/phase3'
PROCESSED = PROJECT_ROOT / 'data/processed/cleveland_corral'
print('Project root located; all paths below are repository-relative.')

## Reproduce or verify

A fresh local restart has no ignored processed outputs, so the cell below runs the complete Phase 3 build. When verified processed outputs already exist, it validates and reuses them so repeated notebook execution is fast. Set `FORCE_PHASE3_REBUILD=1` to force a complete rebuild.

In [ ]:
daily_output = PROCESSED / 'phase3_daily_analysis_series.parquet'
force_rebuild = os.environ.get('FORCE_PHASE3_REBUILD') == '1'
if force_rebuild or not daily_output.is_file():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / 'scripts/build_phase3_analysis.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Reusing existing ignored processed outputs; set FORCE_PHASE3_REBUILD=1 to rebuild.')
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts/verify_phase3_outputs.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

## Masks and transformations

Levels require a numeric timestamped observation inside documented operation and outside fatal QC or maintenance flags. Metadata range concerns remain eligible and visible. Changes use consecutive dates inside one installation segment; cumulative displacement also stays inside one water year. Rain is defined two ways: valid within-water-year differences of the published cumulative daily field, and sums of at least 90 eligible 15-minute interval-rain values.

In [ ]:
configuration = pd.read_csv(TABLES / 'analysis_configuration.csv')
display(configuration)

## Coverage, missingness, and regimes

A blank measurement is a published station row with no sensor value. An absent timestamp is a missing expected row inside a segment span. They have different implications and are never converted to zero.

In [ ]:
coverage = pd.read_csv(TABLES / 'coverage_missingness.csv')
display(
    coverage.loc[
        coverage['product_type'].eq('daily'),
        ['sensor_id', 'installation_segment_id', 'absent_timestamp_count',
         'blank_measurement_count', 'eligible_level_count',
         'range_concern_retained_count'],
    ]
)
display(Image(filename=str(FIGURES / '01_daily_coverage_missingness.png')))

## Trend, seasonality, and stationarity

STL is used only on exact contiguous daily runs at least two annual cycles long, with 365 days selected and 366 days checked. ADF tests a unit-root null; KPSS tests a level-stationarity null. Agreement is more informative than either test alone, while seasonality and structural behavior still limit both.

In [ ]:
stationarity = pd.read_csv(TABLES / 'stationarity_diagnostics.csv')
summary = (
    stationarity.groupby(['sensor_id', 'transformation', 'joint_interpretation'])
    .size()
    .rename('run_count')
    .reset_index()
)
display(summary)
display(Image(filename=str(FIGURES / '04_water_year_seasonality.png')))
display(Image(filename=str(FIGURES / '05_robust_stl_decomposition.png')))

## ACF and PACF

ACF measures correlation between a series and lagged copies. PACF measures the additional association at a lag after accounting for shorter lags. The confidence bands here are approximate; autocorrelation, seasonality, and finite samples complicate their interpretation. The synthetic demonstration uses seed 170 and is never mixed with USGS observations.

In [ ]:
display(Image(filename=str(FIGURES / '06_daily_acf_pacf.png')))
display(Image(filename=str(FIGURES / '09_synthetic_linear_processes.png')))

## Exact-date daily lag associations

**Positive lag means the predictor leads the response.** The declared search is 0–30 days. Naive level correlations are compared with valid changes and cautious predictor-AR(1) prewhitening. This is essential because shared trend, seasonality, and autocorrelation can create apparently strong cross-correlations without a physical lag.

In [ ]:
lags = pd.read_csv(TABLES / 'daily_lag_summary.csv')
display(
    lags.loc[
        lags['method'].isin(['transformed', 'prewhitened']),
        ['window_id', 'predictor', 'response', 'method', 'rain_definition',
         'peak_lag_days', 'peak_correlation', 'peak_pair_count',
         'conditional_peak_correlation_ci_low',
         'conditional_peak_correlation_ci_high'],
    ]
)
display(Image(filename=str(FIGURES / '07_daily_lag_sensitivity.png')))

## Rain-selected 15-minute events

Events are chosen from rain totals, never displacement outcomes. Shifted rain is matched one-to-one to response changes without reuse or interpolation. Eight- and 15-minute tolerances test clock-alignment sensitivity. Stability across tolerances does not imply stability across storms.

In [ ]:
events = pd.read_csv(TABLES / 'event_selection.csv')
event_sensitivity = pd.read_csv(TABLES / 'event_alignment_sensitivity.csv')
display(events)
display(event_sensitivity)
display(Image(filename=str(FIGURES / '08_event_alignment_sensitivity.png')))

## Interpretation boundary

Observed results are the published values, masks, and aggregate diagnostics. Statistical interpretation concerns stationarity, dependence, and lag sensitivity. Engineering interpretation may compare the timing with the precipitation → wetting/pressure → movement hypothesis. Speculation about flow paths or mechanisms remains untested. Temporal association does not establish causation, and nothing here is an operational-warning or design result.

See `docs/CLEVELAND_CORRAL_PHASE3_EXPLORATORY_DYNAMICS.md` for the complete findings, limitations, and exact recommended Phase 4 objective.